# Steel rebar physical-market prices

This notebook reads the immutable IME rebar raw CSV and the reproducible A3 / 12 mm daily derived dataset. It plots daily volume-weighted **cash trade price** (`Price`) and **offer-base price** (`ArzeBasePrice`) for that working scope. Dates remain in the source Jalali calendar.

> **Important:** the chart scope is plainly specified straight A3 / 12 mm rebar under cash or cash-matching contracts. It is still exploratory—not an approved comparable-rebar benchmark—because producer, delivery, standard and quotation-basis comparability remain to be validated. The source `Unit` records traded quantity (tonnes).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT = Path.cwd().resolve().parent
WORKSPACE = PROJECT.parents[1]
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from shared.ime_data.ime_physical_collector import normalize_fa

RAW_PATH = PROJECT / 'data' / 'raw' / 'physical' / 'rebar_physical_raw.csv'
A3_12_DAILY_PATH = PROJECT / 'data' / 'processed' / 'rebar_a3_12_cash_daily.csv'
FIGURE_PATH = PROJECT / 'outputs' / 'figures' / 'rebar_cash_vs_offer_base_price.png'
RAW_PATH

In [ ]:
if not RAW_PATH.exists():
    raise FileNotFoundError(f'Run the physical collector first: {RAW_PATH}')

raw = pd.read_csv(RAW_PATH, encoding='utf-8-sig', low_memory=False)
for column in ['Price', 'ArzeBasePrice', 'Quantity']:
    raw[column] = pd.to_numeric(
        raw[column].astype(str).str.replace(',', '', regex=False), errors='coerce'
    )

raw['trade_date_jalali'] = raw['date'].astype(str).str.replace('-', '/', regex=False)
raw['contract_type_normalized'] = raw['ContractType'].map(normalize_fa)
from commodity.rebar.src.rebar.processing.rebar_scope import CASH_CONTRACTS

cash_contracts = CASH_CONTRACTS
cash = raw.loc[raw['contract_type_normalized'].isin(cash_contracts)].copy()
cash = cash.loc[
    cash['Quantity'].gt(0) & cash['Price'].gt(0) & cash['ArzeBasePrice'].gt(0)
].copy()

assert cash['trade_date_jalali'].notna().all()
assert cash['Quantity'].gt(0).all()
assert cash['Price'].gt(0).all()
assert cash['ArzeBasePrice'].gt(0).all()

print(f'Raw rows: {len(raw):,}')
print(f'Eligible cash rows: {len(cash):,}')
print(f'Jalali coverage: {cash.trade_date_jalali.min()} to {cash.trade_date_jalali.max()}')
print('Source quantity units:', sorted(raw['Unit'].dropna().astype(str).unique()))

In [ ]:
from commodity.rebar.src.rebar.processing.rebar_scope import is_a3_12_straight_rebar

a3_12_source = raw.loc[
    raw['GoodsName'].map(is_a3_12_straight_rebar)
    & raw['contract_type_normalized'].isin(cash_contracts)
    & raw['Quantity'].gt(0)
    & raw['Price'].gt(0)
    & raw['ArzeBasePrice'].gt(0)
].copy()

assert not a3_12_source.empty
assert a3_12_source['GoodsName'].map(is_a3_12_straight_rebar).all()
assert a3_12_source['contract_type_normalized'].isin(cash_contracts).all()
print(f'Strict A3 / 12 mm source rows selected before aggregation: {len(a3_12_source):,}')
print(f'Cash-trade dates before aggregation: {a3_12_source.trade_date_jalali.nunique():,}')
a3_12_source[['GoodsName', 'Symbol', 'ProducerName', 'ContractType']].value_counts().head(20)

In [ ]:
if not A3_12_DAILY_PATH.exists():
    raise FileNotFoundError(
        'Build the selected scope first: python .\commodity\rebar\src\rebar\processing\build_a3_12_cash_daily.py'
    )
a3_12_daily = pd.read_csv(A3_12_DAILY_PATH, encoding='utf-8-sig')
assert a3_12_daily['trade_date_jalali'].is_unique
assert a3_12_daily['traded_quantity'].gt(0).all()

plt.style.use('seaborn-v0_8-whitegrid')
fig, (price_ax, gap_ax) = plt.subplots(2, 1, figsize=(16, 11), sharex=True, constrained_layout=True)
ticks = list(range(0, len(a3_12_daily), max(1, len(a3_12_daily) // 14)))
if ticks[-1] != len(a3_12_daily) - 1:
    ticks.append(len(a3_12_daily) - 1)

price_ax.plot(
    a3_12_daily['trade_date_jalali'], a3_12_daily['cash_trade_price_vwap'],
    color='#0b5fa5', linewidth=1.3, label='Cash trade price (VWAP; Price)'
)
price_ax.plot(
    a3_12_daily['trade_date_jalali'], a3_12_daily['offer_base_price_vwap'],
    color='#d45d00', linewidth=1.3, label='Offer-base price (VWAP; ArzeBasePrice)'
)
price_ax.set_title('IME straight rebar A3 / 12 mm cash contracts: daily price levels')
price_ax.set_ylabel('IME quoted price (source basis pending validation)')
price_ax.legend(frameon=True)
price_ax.margins(x=0)

gap_ax.axhline(0, color='black', linewidth=0.9)
gap_ax.plot(
    a3_12_daily['trade_date_jalali'], a3_12_daily['cash_vs_offer_base_pct'],
    color='#6a3d9a', linewidth=1.15, label='Cash price relative to offer-base price'
)
gap_ax.fill_between(
    range(len(a3_12_daily)), a3_12_daily['cash_vs_offer_base_pct'], 0,
    where=a3_12_daily['cash_vs_offer_base_pct'].ge(0), color='#2a9d8f', alpha=0.25, label='Cash above base'
)
gap_ax.fill_between(
    range(len(a3_12_daily)), a3_12_daily['cash_vs_offer_base_pct'], 0,
    where=a3_12_daily['cash_vs_offer_base_pct'].lt(0), color='#e76f51', alpha=0.25, label='Cash below base'
)
gap_ax.set_title('Daily percentage difference: 100 × (cash trade VWAP / offer-base VWAP − 1)')
gap_ax.set_ylabel('Percent')
gap_ax.set_xlabel('Trade date (Jalali)')
gap_ax.set_xticks(ticks, a3_12_daily['trade_date_jalali'].iloc[ticks], rotation=45, ha='right')
gap_ax.legend(loc='best', ncol=3, frameon=True)
gap_ax.margins(x=0)

FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_PATH, dpi=180, bbox_inches='tight')
plt.show()
print(f'A3 / 12 mm cash dates: {len(a3_12_daily):,}')
print(f'Median daily gap: {a3_12_daily.cash_vs_offer_base_pct.median():.2f}%')
print(f'Saved: {FIGURE_PATH}')

## Screening candidate homogeneous products

The raw goods names are highly heterogeneous and include mixed baskets, coil, alloy, and short-length products. For an **initial screen**, the next cell retains only plainly specified straight, single-diameter, single-grade rebar labels. It maps equivalent grade-before- and grade-after-diameter spellings to one key, such as `A3 / 16 mm`.

This screening result is not a final benchmark selection. Producer, standard, delivery location, and contract comparability must be assessed before economic interpretation.

In [ ]:
from commodity.rebar.src.rebar.processing.rebar_scope import canonical_straight_rebar_label

screened = raw.copy()
screened['canonical_product'] = screened['GoodsName'].map(canonical_straight_rebar_label)
screened = screened.loc[screened['canonical_product'].notna()].copy()
screened['is_positive_trade'] = screened['Quantity'].gt(0) & screened['Price'].gt(0)
screened['is_cash_positive_trade'] = (
    screened['is_positive_trade'] & screened['contract_type_normalized'].isin(cash_contracts)
)

def count_days(frame: pd.DataFrame, mask_column: str) -> int:
    return frame.loc[frame[mask_column], 'trade_date_jalali'].nunique()

candidate_summary = (
    screened.groupby('canonical_product', sort=False)
    .apply(
        lambda frame: pd.Series({
            'source_rows': len(frame),
            'positive_trade_rows': int(frame['is_positive_trade'].sum()),
            'positive_trade_days': count_days(frame, 'is_positive_trade'),
            'cash_positive_rows': int(frame['is_cash_positive_trade'].sum()),
            'cash_positive_days': count_days(frame, 'is_cash_positive_trade'),
            'positive_traded_quantity': frame.loc[frame['is_positive_trade'], 'Quantity'].sum(),
            'producer_count': frame.loc[frame['is_positive_trade'], 'ProducerName'].nunique(),
            'first_positive_date': frame.loc[frame['is_positive_trade'], 'trade_date_jalali'].min(),
            'last_positive_date': frame.loc[frame['is_positive_trade'], 'trade_date_jalali'].max(),
        }),
        include_groups=False,
    )
    .sort_values(['cash_positive_rows', 'cash_positive_days', 'positive_traded_quantity'], ascending=False)
)

print(f'Straight single-diameter/single-grade source rows retained: {len(screened):,}')
candidate_summary.head(15)

## Construction notes

- Eligible rows have a source cash or cash-matching `ContractType`, positive executed quantity, positive trade price, and positive offer-base price.
- Both plotted series use the same executed-quantity weights, so their daily difference is not driven by different row weighting. The lower panel calculates `100 × (cash trade VWAP / offer-base VWAP − 1)`: positive values indicate cash trades above the corresponding offer base.
- The notebook explicitly verifies the strict A3 / 12 mm filter before reading or plotting any daily aggregation. The chart never calculates a VWAP across the broad rebar-labelled raw universe.
- The product ranking is a transparent screening rule, not a final economic eligibility decision. It ranks canonical diameter/grade groups by positive cash-trade rows, then positive cash-trade days, then quantity. The derived daily output preserves source goods-name, contract-type, symbol, and producer audit fields.